## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE = '/content/drive/MyDrive/ras2026_cache'
os.makedirs(DRIVE_CACHE, exist_ok=True)

if not os.path.exists('ras2026'):
    !git clone https://github.com/nepersoned/ras2026.git
os.chdir('ras2026')
!git pull
print('cwd:', os.getcwd())

In [ ]:
!pip install -q pybind11 ortools scipy numpy pandas torch joblib scikit-learn

In [ ]:
DRIVE_DATA = '/content/drive/MyDrive/ras_release_v1.1'
if not os.path.exists('ras_release_v1.1'):
    os.symlink(DRIVE_DATA, 'ras_release_v1.1')
print('Data ready:', os.path.exists('ras_release_v1.1'))

## 1. Build C++ Modules

In [ ]:
import subprocess, sys

pybind_inc = subprocess.check_output(
    [sys.executable, '-m', 'pybind11', '--includes']
).decode().strip()
ext_suffix = subprocess.check_output(
    [sys.executable, '-c', 'import sysconfig; print(sysconfig.get_config_var("EXT_SUFFIX"))']
).decode().strip()

modules = [
    ('evaluator',    'src/evaluator.cpp',    '-O3'),
    ('cg_pricer_v2', 'src/cg_pricer_v2.cpp', '-O3 -fopenmp'),
]
for name, src, flags in modules:
    out = f'src/{name}{ext_suffix}'
    cmd = f'g++ {flags} -shared -fPIC {pybind_inc} {src} -o {out}'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'{name}: OK')
    else:
        print(f'{name}: ERROR')
        print(result.stdout + result.stderr)

In [ ]:
import sys
sys.path.insert(0, 'src')
sys.path.insert(0, '.')

import math, random, time, json, csv
from pathlib import Path
from copy import deepcopy
from collections import defaultdict, Counter

import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import AgglomerativeClustering

import evaluator as ev
import cg_pricer_v2 as cgp

from solver import (
    load_layer, load_od_matrix, Network, Demand, Solution,
    greedy_construct, evaluate, build_json,
    COMMODITY_TO_BLOCK_TYPE, DIRECT_ONLY,
)
from src.repair import ortools_repair

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 2. Environment + joblib Cache

In [ ]:
COMMODITY_IDX    = {'Manifest': 0, 'Bulk': 1, 'Intermodal': 2, 'Multilevel': 3}
DIRECT_ONLY_COMM = {'Intermodal', 'Automobile'}
MAX_HUBS = 50
FEAT_DIM = 7 + MAX_HUBS * 4


def load_env(layer, multiplier):
    cache_jl  = f'{DRIVE_CACHE}/od_matrix_{layer}.joblib'
    cache_pkl = f'{DRIVE_CACHE}/od_matrix_{layer}.pkl'
    net_cache = f'{DRIVE_CACHE}/network_{layer}.joblib'

    nodes_df, links_df, demands_scaled, demands_raw, settings = load_layer(layer, multiplier)
    yard_rows = nodes_df[nodes_df['node_type'] == 'yard']
    yard_info = {
        int(r['node_id']): {
            'num_tracks':        float(r.get('num_tracks', 9999) or 9999),
            'handling_capacity': float(r.get('handling_capacity', 1e9) or 1e9),
            'handling_cost':     float(r.get('handling_cost', 0) or 0),
        } for _, r in yard_rows.iterrows()
    }
    origin_ids   = set(demands_scaled['origin_yard_id'].astype(int))
    dest_ids     = set(demands_scaled['dest_yard_id'].astype(int))
    all_yard_ids = sorted(origin_ids | dest_ids)

    # ── od_matrix + Network: 둘 다 캐시 있으면 즉시 로드 ──────────────────────
    if os.path.exists(cache_jl) and os.path.exists(net_cache):
        t0 = time.time()
        print(f'  Loading od_matrix + Network from joblib cache...')
        od_matrix = joblib.load(cache_jl)
        net       = joblib.load(net_cache)
        print(f'  Loaded in {time.time()-t0:.1f}s')

    else:
        # od_matrix 로드 또는 계산
        if os.path.exists(cache_jl):
            od_matrix = joblib.load(cache_jl)
        elif os.path.exists(cache_pkl):
            import pickle
            print(f'  Migrating od_matrix pickle → joblib...')
            t0 = time.time()
            with open(cache_pkl, 'rb') as f:
                od_matrix = pickle.load(f)
            print(f'  Loaded in {time.time()-t0:.1f}s, saving joblib...')
            joblib.dump(od_matrix, cache_jl, compress=3)
        else:
            print(f'  Computing od_matrix (first time)...')
            all_od_pairs = {(o,d) for o in all_yard_ids for d in all_yard_ids if o!=d}
            od_matrix    = load_od_matrix(all_od_pairs)
            joblib.dump(od_matrix, cache_jl, compress=3)
            print(f'  Saved {cache_jl}')

        # Network Dijkstra 계산 후 캐시
        print(f'  Building Network (Dijkstra) — will cache for next run...')
        t0 = time.time()
        net = Network(nodes_df, links_df, origin_ids, dest_ids, settings, verbose=True)
        print(f'  Network built in {time.time()-t0:.1f}s, caching...')
        try:
            joblib.dump(net, net_cache, compress=3)
            print(f'  Network cached → next run instant')
        except Exception as e:
            print(f'  Network cache failed (will recompute next time): {e}')

    demands = [
        Demand(
            idx=idx, demand_id=int(r['demand_id']),
            commodity_type=str(r['block_type']),
            block_type=COMMODITY_TO_BLOCK_TYPE.get(str(r['block_type']), 'Manifest'),
            origin=int(r['origin_yard_id']), dest=int(r['dest_yard_id']),
            volume=int(r['volume']),
            sp_dist=od_matrix.get(
                (int(r['origin_yard_id']), int(r['dest_yard_id'])),
                net.dist(int(r['origin_yard_id']), int(r['dest_yard_id']))
            ),
        ) for idx, (_, r) in enumerate(demands_scaled.iterrows())
    ]
    return dict(net=net, od_matrix=od_matrix, settings=settings,
                yard_info=yard_info, demands=demands, all_yards=all_yard_ids,
                nodes_df=nodes_df, links_df=links_df, demands_raw=demands_raw)


def ranked_hubs(dem, env):
    if dem.commodity_type in DIRECT_ONLY_COMM:
        return []
    scored = []
    for hub in env['all_yards']:
        if hub in (dem.origin, dem.dest): continue
        d1 = env['od_matrix'].get((dem.origin, hub), math.inf)
        d2 = env['od_matrix'].get((hub, dem.dest), math.inf)
        if math.isinf(d1) or math.isinf(d2): continue
        scored.append((d1+d2, hub))
    scored.sort()
    return [h for _,h in scored]


def _build_cands(dem, hubs, env):
    s   = env['settings']
    tc  = float(s.get('transport_cost_coefficient', 1.0))
    ic  = float(s.get('interchange_cost', 100))
    M   = float(s.get('stress_penalty_M', 5))
    net = env['net']
    cands = [(None, M * dem.volume * dem.sp_dist)]
    od_d = net.dist(dem.origin, dem.dest)
    if not math.isinf(od_d):
        od_ic = net.interchanges(dem.origin, dem.dest)
        cands.append(([(dem.origin, dem.dest)],
                      tc*dem.volume*od_d + ic*dem.volume*od_ic))
    if dem.commodity_type not in DIRECT_ONLY_COMM:
        for hub in hubs[:MAX_HUBS]:
            d1 = net.dist(dem.origin, hub)
            d2 = net.dist(hub, dem.dest)
            if math.isinf(d1) or math.isinf(d2): continue
            ic1 = net.interchanges(dem.origin, hub)
            ic2 = net.interchanges(hub, dem.dest)
            hc  = env['yard_info'].get(hub,{}).get('handling_cost',0.0)
            cands.append(([(dem.origin,hub),(hub,dem.dest)],
                          tc*dem.volume*(d1+d2) + ic*dem.volume*(ic1+ic2) + hc*dem.volume))
    return cands


def precompute(env):
    data = []
    for dem in env['demands']:
        if dem.volume <= 0:
            data.append(None); continue
        hubs  = ranked_hubs(dem, env)
        feat  = _demand_feat(dem, hubs, env)
        cands = _build_cands(dem, hubs, env)
        data.append({'feat': feat, 'hubs': hubs, 'candidates': cands,
                     'direct_only': dem.commodity_type in DIRECT_ONLY_COMM})
    return data


def _demand_feat(dem, hubs, env):
    od_d  = env['net'].dist(dem.origin, dem.dest)
    od_ic = env['net'].interchanges(dem.origin, dem.dest)
    bt_oh = [1.0 if COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest')==t else 0.0
             for t in ['Manifest','Bulk','Intermodal','Multilevel']]
    base = [math.log1p(dem.volume),
            math.log1p(od_d if math.isfinite(od_d) else 1e6),
            od_ic/5.0] + bt_oh
    hub_f = [0.0]*(MAX_HUBS*4)
    for j, hub in enumerate(hubs[:MAX_HUBS]):
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2): continue
        ic1 = env['net'].interchanges(dem.origin, hub)
        ic2 = env['net'].interchanges(hub, dem.dest)
        hc  = env['yard_info'].get(hub,{}).get('handling_cost',0.0)
        hub_f[j*4]=math.log1p(d1); hub_f[j*4+1]=math.log1p(d2)
        hub_f[j*4+2]=(ic1+ic2)/5.0; hub_f[j*4+3]=hc/500.0
    return base + hub_f


print('Environment ready.')

## 3. True Column Generation

In [ ]:
from ortools.linear_solver import pywraplp


def cg_solve_v2(env, dd, max_iter=50, time_limit_s=180, verbose=True):
    """
    True Column Generation:
    - Master LP with OR-Tools GLOP
    - Pricing via C++ cg_pricer_v2: searches ALL yards as hubs
    - Adds genuinely new routes not in precomputed candidate list
    """
    demands = env['demands']
    s       = env['settings']
    tc      = float(s.get('transport_cost_coefficient', 1.0))
    M_pen   = float(s.get('stress_penalty_M', 5))
    n       = len(demands)
    t_start = time.time()

    # Build od_dist map for C++ pricer (string keys)
    od_dist_cpp = {f'{o}_{d}': float(dist)
                   for (o,d), dist in env['od_matrix'].items()
                   if math.isfinite(dist)}
    yard_hc_cpp = {k: float(v['handling_cost']) for k,v in env['yard_info'].items()}
    yard_ids    = env['all_yards']

    # demands_cpp: (idx, origin, dest, volume, direct_only)
    demands_cpp = [
        (dem.idx, dem.origin, dem.dest, dem.volume,
         dem.commodity_type in DIRECT_ONLY_COMM)
        for dem in demands
    ]

    # columns[i] = list of (route, cost) — starts from precomputed candidates
    columns = []
    for i, d in enumerate(dd):
        if d is None:
            columns.append([(None, M_pen * demands[i].volume * demands[i].sp_dist)])
        else:
            columns.append(list(d['candidates']))

    best_obj      = float('inf')
    best_routings = None

    for it in range(max_iter):
        if time.time() - t_start > time_limit_s:
            if verbose: print(f'  CG: time limit reached at iter {it}')
            break

        # ── Master LP ────────────────────────────────────────────────────────
        solver = pywraplp.Solver.CreateSolver('GLOP')
        solver.SuppressOutput()

        x = {}
        for i in range(n):
            for ri in range(len(columns[i])):
                x[(i, ri)] = solver.NumVar(0.0, 1.0, f'x_{i}_{ri}')

        for i in range(n):
            ct = solver.Constraint(1.0, 1.0, f'd_{i}')
            for ri in range(len(columns[i])):
                ct.SetCoefficient(x[(i, ri)], 1.0)

        obj = solver.Objective()
        obj.SetMinimization()
        for (i, ri), var in x.items():
            obj.SetCoefficient(var, columns[i][ri][1])

        status = solver.Solve()
        if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
            break

        lp_obj = solver.Objective().Value()

        # Extract duals
        duals = [0.0] * n
        for i in range(n):
            ct = solver.LookupConstraint(f'd_{i}')
            if ct: duals[i] = ct.dual_value()

        # ── True CG Pricing: C++ searches ALL hubs ────────────────────────────
        new_cols = cgp.price_full_hubs(
            od_dist_cpp, yard_ids, yard_hc_cpp,
            demands_cpp, duals, tc, M_pen, tol=1e-4
        )

        added = 0
        for nc in new_cols:
            i   = nc.demand_idx
            hub = nc.hub
            cost= nc.cost
            if hub == -1:
                route = [(demands[i].origin, demands[i].dest)]
            else:
                route = [(demands[i].origin, hub), (hub, demands[i].dest)]
            # Check not duplicate
            if not any(c[0] == route for c in columns[i]):
                columns[i].append((route, cost))
                added += 1

        # Round LP → integer solution
        routings = []
        for i in range(n):
            best_ri = max(range(len(columns[i])),
                          key=lambda ri: x[(i,ri)].solution_value())
            routings.append(columns[i][best_ri][0])

        if lp_obj < best_obj:
            best_obj      = lp_obj
            best_routings = routings

        if verbose:
            print(f'  CG iter {it+1:3d} | LP={lp_obj:>18,.0f} | '
                  f'+cols={added:5d} | {time.time()-t_start:.1f}s')

        if added == 0:
            if verbose: print(f'  CG converged.')
            break

    # Update dd with new columns found by CG
    for i, d in enumerate(dd):
        if d is not None:
            d['candidates'] = columns[i]

    if best_routings is None:
        sol = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                               env['settings'], env['yard_info'])
        best_routings = sol.routings

    return best_routings


print('True CG ready.')

## 4. AISO v2 — Stochastic Destroy

In [ ]:
def build_smart_M(mu, K, gamma=0.5):
    mu_min, mu_max = mu.min(), mu.max()
    if mu_max - mu_min < 1e-8:
        return np.zeros((K, K))
    mu_n = (mu - mu_min) / (mu_max - mu_min)
    M = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            if i != j:
                M[i, j] = gamma * (mu_n[j] - mu_n[i])
    return M


class AISODestroyV2:
    """
    AISO v2 destroy operator:
    - Per-episode W noise → breaks deterministic loop
    - Softmax sampling (not hard top-K) → stochastic diverse selection
    - Smart M: antisymmetric volume gradient
    """
    def __init__(self, dd, env, K=10, N_agents=20, T_iters=30, gamma=0.5):
        demands = env['demands']
        self.K  = K
        self.N  = N_agents
        self.T  = T_iters
        self.alpha = 0.4
        self.beta  = 0.1

        # Cluster demands
        all_yards = sorted(env['all_yards'])
        Y = len(all_yards)
        yard_to_idx = {y: i for i, y in enumerate(all_yards)}

        feats, valid = [], []
        for i, d in enumerate(dd):
            if d is None: continue
            dem = demands[i]
            feats.append([
                yard_to_idx.get(dem.origin, 0) / max(Y, 1),
                yard_to_idx.get(dem.dest, 0)   / max(Y, 1),
                math.log1p(dem.volume) / 15.0,
                1.0 if dem.commodity_type in DIRECT_ONLY_COMM else 0.0,
            ])
            valid.append(i)

        K_use = min(K, max(2, len(feats) // 2))
        self.K = K_use
        X = np.array(feats, dtype=np.float32)
        labels = AgglomerativeClustering(n_clusters=K_use, linkage='ward').fit(X).labels_

        # cluster_of[i] = cluster index, -1 if invalid
        self.cluster_of = [-1] * len(dd)
        self.cluster_demands = defaultdict(list)
        for j, i in enumerate(valid):
            c = int(labels[j])
            self.cluster_of[i] = c
            self.cluster_demands[c].append(i)
        self.valid = valid

        # Per-cluster mean volume → Smart M
        mu = np.array([
            np.mean([demands[i].volume for i in self.cluster_demands.get(k, [0])])
            for k in range(K_use)
        ])
        self.M = build_smart_M(mu, K_use, gamma)
        print(f'  AISO v2: K={K_use} clusters, M {K_use}x{K_use}')
        print(f'  M asymmetry: max|M_ij - M_ji| = {np.max(np.abs(self.M - self.M.T)):.3f}')

        # Initial W
        self.W = np.random.dirichlet(np.ones(K_use), size=N_agents)

    def select(self, K_destroy, route_costs, temperature=1.0, noise=0.1):
        """
        Stochastic demand selection via AISO dynamics.
        temperature: softmax temperature for final sampling
        noise: W perturbation per episode to break determinism
        """
        K, N, T = self.K, self.N, self.T
        valid = self.valid
        if not valid: return []

        # Add noise to W each episode (breaks determinism)
        noise_mat = np.random.dirichlet(np.ones(K), size=N) * noise
        W = self.W + noise_mat
        W = W / W.sum(axis=1, keepdims=True)

        rc = np.zeros(len(route_costs))
        for i in valid:
            rc[i] = route_costs[i]
        rc_max = rc.max() if rc.max() > 0 else 1.0

        # AISO dynamics
        agent_cluster = np.argmax(W, axis=1)  # (N,) each agent's dominant cluster

        for t in range(T):
            # diversity → adaptive repulsion
            dists = [np.linalg.norm(W[i] - W[j])
                     for i in range(N) for j in range(i+1, N)]
            delta = np.mean(dists) if dists else 1.0
            repul = 1.0 + 3.0 * math.exp(-delta / 0.12)

            M_eff = self.M.copy()
            M_eff[M_eff < 0] *= repul

            for i in range(N):
                # Best partner by compatibility × fitness
                scores_j = np.array([
                    float(W[i] @ M_eff @ W[j]) * (rc[self.cluster_demands.get(agent_cluster[j],[0])[0]] / rc_max
                         if self.cluster_demands.get(agent_cluster[j]) else 0.0)
                    if j != i else -np.inf
                    for j in range(N)
                ])
                best_j = int(np.argmax(scores_j))
                c_ij = float(scores_j[best_j])

                # Position update in type space
                new_W = W[i] + self.alpha * c_ij * (W[best_j] - W[i])
                new_W = np.clip(new_W, 1e-8, None)
                new_W /= new_W.sum()

                new_cluster = int(np.argmax(new_W))
                cands = self.cluster_demands.get(new_cluster, valid)
                new_dem_score = np.mean([rc[i2] for i2 in cands]) / rc_max if cands else 0.0
                old_dem_score = np.mean([rc[i2] for i2 in self.cluster_demands.get(agent_cluster[i], valid)]) / rc_max

                if new_dem_score >= old_dem_score:
                    W[i] = new_W
                    agent_cluster[i] = new_cluster
                    # Type assimilation
                    mixed = (1 - self.beta) * W[i] + self.beta * W[best_j]
                    W[i] = mixed / mixed.sum()

        # Update stored W (with momentum)
        self.W = 0.7 * self.W + 0.3 * W
        self.W /= self.W.sum(axis=1, keepdims=True)

        # Aggregate cluster weights
        cluster_w = W.mean(axis=0)  # (K,)

        # Demand scores: cluster_weight × route_cost
        demand_scores = np.zeros(len(rc))
        for i in valid:
            c = self.cluster_of[i]
            w = cluster_w[c] if c >= 0 else 1.0 / K
            demand_scores[i] = w * rc[i]

        # Softmax sampling (stochastic, not hard top-K)
        valid_arr = np.array(valid)
        scores_v  = demand_scores[valid_arr]
        scores_v  = scores_v / (scores_v.max() + 1e-8)
        probs     = np.exp(scores_v / max(temperature, 0.01))
        probs    /= probs.sum()

        K_sample = min(K_destroy, len(valid_arr))
        chosen   = np.random.choice(len(valid_arr), size=K_sample, replace=False, p=probs)
        return valid_arr[chosen].tolist()


print('AISO v2 ready.')

## 5. C++ Solver Setup

In [ ]:
def build_cpp_solver(env, dd, init_routings, verbose=False):
    demands = env['demands']
    s = env['settings']

    dem_list = []
    for i, dem in enumerate(demands):
        comm_idx = COMMODITY_IDX.get(COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest'), 0)
        dem_list.append((i, dem.origin, dem.dest, dem.volume, dem.sp_dist, comm_idx,
                         dem.commodity_type in DIRECT_ONLY_COMM))

    cand_list = []
    for i, d in enumerate(dd):
        if d is None:
            cand_list.append([(True, False, -1, 0.0, 0.0, -1, -1, -1, -1)])
            continue
        dem = demands[i]
        cpp_cands = []
        for route, cost in d['candidates']:
            if route is None:
                cpp_cands.append((True, False, -1, cost, 0.0, -1, -1, -1, -1))
            elif len(route) == 1:
                o, dst = route[0]
                cpp_cands.append((False, True, -1, cost, 0.0, o, dst, -1, -1))
            else:
                (o1,h),(h2,d2) = route
                hc = env['yard_info'].get(h,{}).get('handling_cost',0.0) * dem.volume
                cpp_cands.append((False, False, h, cost-hc, hc, o1, h, h, d2))
        cand_list.append(cpp_cands)

    init_route_idx = []
    for i, routing in enumerate(init_routings):
        d = dd[i]
        if d is None: init_route_idx.append(0); continue
        found = 0
        for ri, (route, _) in enumerate(d['candidates']):
            if route == routing: found = ri; break
        init_route_idx.append(found)

    yd_tracks = {k: int(v['num_tracks']) for k,v in env['yard_info'].items()}
    yd_hcap   = {k: float(v['handling_capacity']) for k,v in env['yard_info'].items()}
    seg_dist  = {f'{o}_{d}': float(dist)
                 for (o,d), dist in env['od_matrix'].items() if math.isfinite(dist)}

    cpp_s = ev.Settings()
    cpp_s.block_fixed_cost     = float(s.get('block_fixed_cost', 1500))
    cpp_s.unserved_penalty     = float(s.get('stress_penalty_M', 5))
    cpp_s.min_block_vol_short  = 5.0
    cpp_s.min_block_vol_medium = 10.0
    cpp_s.min_block_vol_long   = 15.0

    solver = ev.RasSolver()
    solver.init(dem_list, cand_list, init_route_idx,
                yd_tracks, yd_hcap, seg_dist, cpp_s)
    if verbose:
        print(f'  C++ init: {solver.get_score():,.0f}')
    return solver


def routes_to_routings(route_indices, dd, env):
    routings = []
    for i, ri in enumerate(route_indices):
        d = dd[i]
        if d is None: routings.append(None); continue
        routings.append(d['candidates'][ri][0])
    return routings


print('C++ solver ready.')

## 6. Main Training Loop

In [ ]:
def cg_v2_train(
    env, dd,
    n_episodes=150,
    sa_iters_per_ep=5000,
    T0_frac=0.15,         # wider than v1 (was 0.05)
    T_final_frac=0.0005,
    K_destroy=10,
    K_clusters=10,
    aiso_agents=20,
    aiso_iters=40,
    aiso_noise=0.15,
    aiso_temp_init=2.0,
    aiso_temp_final=0.3,
    final_sa_iters=200000,
    cg_max_iter=50,
    cg_time_limit=180,
    print_every=25,
):
    t_total = time.time()

    # ── Step 1: True CG ──────────────────────────────────────────────────────
    print('\n── CG v2 (true pricing, all hubs) ──')
    cg_routings = cg_solve_v2(env, dd, max_iter=cg_max_iter,
                              time_limit_s=cg_time_limit, verbose=True)

    cpp_solver = build_cpp_solver(env, dd, cg_routings, verbose=True)
    cg_score   = cpp_solver.get_score()

    # Fallback to greedy if CG is worse
    greedy_sol   = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                    env['settings'], env['yard_info'])
    greedy_score, _ = evaluate(greedy_sol, env['net'], env['od_matrix'],
                               env['settings'], env['yard_info'])
    if cg_score > greedy_score * 1.01:
        print(f'  CG({cg_score:,.0f}) > greedy({greedy_score:,.0f}), using greedy')
        cpp_solver = build_cpp_solver(env, dd, greedy_sol.routings)
        cg_score   = cpp_solver.get_score()
    else:
        print(f'  CG score: {cg_score:,.0f}  (greedy: {greedy_score:,.0f}, '
              f'gain: {(greedy_score-cg_score)/greedy_score*100:.1f}%)')

    # ── Step 2: AISO v2 destroy operator ─────────────────────────────────────
    print('\n── AISO v2 ──')
    aiso = AISODestroyV2(dd, env, K=K_clusters,
                         N_agents=aiso_agents, T_iters=aiso_iters)

    # ── Step 3: AISO-LNS+SA ──────────────────────────────────────────────────
    best_score  = cpp_solver.get_score()
    best_routes = list(cpp_solver.get_routes())
    T0          = best_score * T0_frac
    T_end       = best_score * T_final_frac

    print(f'\n── AISO-LNS+SA ──  init={best_score:,.0f}  ep={n_episodes}')
    print(f'  SA T0={T0:,.0f} → T_end={T_end:,.0f}  K={K_destroy}')

    log = []
    for ep in range(1, n_episodes+1):
        frac     = ep / n_episodes
        T_ep     = T0 * ((T_end/T0) ** frac)
        # AISO temperature anneals (diverse early, focused late)
        aiso_tmp = aiso_temp_init * ((aiso_temp_final/aiso_temp_init) ** frac)

        # ── AISO selects K demands to destroy ────────────────────────────────
        route_idx   = list(cpp_solver.get_routes())
        route_costs = np.zeros(len(dd))
        valid_idx   = []
        for i, d in enumerate(dd):
            if d is None: continue
            ri = route_idx[i]
            route_costs[i] = d['candidates'][ri][1]
            valid_idx.append(i)

        top_k = aiso.select(K_destroy, route_costs,
                            temperature=aiso_tmp, noise=aiso_noise)

        # Weighted SA: focus on AISO-selected demands
        weights = [1e-6] * len(dd)
        for i in top_k:
            weights[i] = 1.0 / max(len(top_k), 1)

        # C++ SA
        score_before = cpp_solver.get_score()
        sa_result = cpp_solver.sa_run(
            weights,
            float(T_ep), float(T_ep * 0.01),
            int(sa_iters_per_ep), int(sa_iters_per_ep // 10),
            int(ep),
        )
        cpp_solver.set_routes(sa_result.best_routes)

        # OR-Tools repair
        if top_k:
            cur_routings = routes_to_routings(list(cpp_solver.get_routes()), dd, env)
            repaired     = ortools_repair(top_k, dd, env, cur_routings)
            cpp2 = build_cpp_solver(env, dd, repaired)
            if cpp2.get_score() < cpp_solver.get_score():
                cpp_solver = cpp2

        score_after = cpp_solver.get_score()
        if score_after < best_score:
            best_score  = score_after
            best_routes = list(cpp_solver.get_routes())

        log.append({'ep': ep, 'score': score_after, 'best': best_score})
        if ep % print_every == 0:
            print(f'  ep {ep:4d} | {score_after:>16,.0f} | best={best_score:>16,.0f} '
                  f'| T={T_ep:.0f}')

    # ── Step 4: Final SA polishing ────────────────────────────────────────────
    print(f'\n── Final SA ({final_sa_iters:,} iters) ──')
    cpp_solver.set_routes(best_routes)
    uniform_w = [1.0 / max(len(dd), 1)] * len(dd)
    pol = cpp_solver.sa_run(
        uniform_w,
        float(best_score * 0.10), float(best_score * 0.0001),
        int(final_sa_iters), int(final_sa_iters // 10),
        999,
    )
    if pol.best_score < best_score:
        best_score  = pol.best_score
        best_routes = list(pol.best_routes)
        print(f'  Improved: {best_score:,.0f}')
    else:
        print(f'  No improvement. best={best_score:,.0f}')

    cpp_solver.set_routes(best_routes)
    best_routings = routes_to_routings(best_routes, dd, env)
    print(f'  Total: {time.time()-t_total:.1f}s')
    return best_routings, best_score, log


print('Training loop ready.')

## 7. Run: L1 → L2 → L3

In [ ]:
CASE_ORDER = [
    ('l1', 0.5), ('l1', 1.0), ('l1', 2.0),
    ('l2', 0.5), ('l2', 1.0), ('l2', 2.0),
    ('l3', 0.5), ('l3', 1.0), ('l3', 2.0),
]

HP = {
    'l1': dict(n_episodes=200, sa_iters_per_ep=5000,  K_destroy=5,
               K_clusters=8,  T0_frac=0.15, cg_max_iter=50, cg_time_limit=60,
               final_sa_iters=300000),
    'l2': dict(n_episodes=100, sa_iters_per_ep=10000, K_destroy=20,
               K_clusters=10, T0_frac=0.15, cg_max_iter=30, cg_time_limit=120,
               final_sa_iters=500000),
    'l3': dict(n_episodes=50,  sa_iters_per_ep=20000, K_destroy=50,
               K_clusters=15, T0_frac=0.15, cg_max_iter=10, cg_time_limit=180,
               final_sa_iters=1000000),
}

solutions  = {}
train_logs = {}
print('Hyperparameters set.')

## 7. Load L1+L2 Solutions from Drive

In [ ]:
import joblib
LOAD_PATH = f"{DRIVE_CACHE}/solutions_l1l2.joblib"
solutions = joblib.load(LOAD_PATH)
train_logs = {}
print(f"Loaded {len(solutions)} solutions from Drive:")
for k,v in solutions.items():
    n_blocks = len(v["outputs"]["1 Block Design"])
    print(f"  {k}: {n_blocks} blocks")


## 8. Run L3

In [ ]:
for mult in [0.5, 1.0, 2.0]:
    tag = f'l3_{mult}'
    print(f'\n{"="*60}\n  L3 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l3', mult)
    dd  = precompute(env)

    best_routings, best_score, log = cg_v2_train(env, dd, **HP['l3'])
    train_logs[tag] = log

    sol = Solution(env['demands'], best_routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')
    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    print(f'  Elapsed: {time.time()-t0:.1f}s')

## 9. Merge All 9 + Submit

In [ ]:
import json, csv
CASE_ORDER = [
    ("l1", 0.5), ("l1", 1.0), ("l1", 2.0),
    ("l2", 0.5), ("l2", 1.0), ("l2", 2.0),
    ("l3", 0.5), ("l3", 1.0), ("l3", 2.0),
]

rows = [["ID", "data"]]
for case_id, (layer, mult) in enumerate(CASE_ORDER):
    tag = f"{layer}_{mult}"
    sol = solutions.get(tag)
    if sol is None:
        print(f"[{case_id}] {tag} MISSING")
        rows.append([case_id, "{}"]); continue
    data_str = json.dumps(sol, separators=(",", ":"))
    print(f"[{case_id}] {tag}  blocks={len(sol["outputs"]["1 Block Design"])}  {len(data_str)/1e6:.1f}MB")
    rows.append([case_id, data_str])

with open("submission_v2_final.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)
print(f"Total: {sum(len(r[1]) for r in rows[1:])/1e6:.1f} MB")

from google.colab import files
files.download("submission_v2_final.csv")
